# Module 9: Importing, Exporting, and Working with Temporary Structures

**ALY 6420 | COPY, \copy, CSV workflows, temporary tables, views, and import validation**

*Course Lecture Notes*


## Module 9

### From querying data to moving data

Up to this point, most of your work has started after the data was already inside PostgreSQL.

Professional analytical work often begins earlier.

A typical workflow looks more like this:

```text
external file / operational system
            ↓
      import or load
            ↓
      validate structure
            ↓
       validate values
            ↓
   transform / analyze / model
            ↓
      export or publish
```

This module focuses on that boundary between PostgreSQL and the outside world.


## Learning objectives

By the end of this lecture, you should be able to:

- explain the role of bulk data movement in an analytical workflow
- distinguish server-side `COPY` from client-side `\copy`
- use `COPY TO` and `COPY FROM` conceptually and correctly
- use CSV-oriented options such as `HEADER`, `DELIMITER`, `NULL`, `QUOTE`, and `ENCODING`
- explain why file location and permissions matter
- distinguish row-by-row `INSERT` from bulk loading
- diagnose common import failures
- create and use temporary tables for staging and intermediate analysis
- create and query standard views
- explain differences among temporary tables, views, materialized views, and permanent tables
- design a validation sequence for a newly imported dataset
- import and verify the ZoomZoom (`sqlda`) dataset for later modules
- explain why successful import does not prove analytical trustworthiness


## Principal source

The principal source for this lecture is:

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 3: Exchanging Data Using COPY**.

Chapter 3 establishes the core data-movement concepts:

- exporting query or table data
- importing large files
- `COPY` versus `\COPY`
- CSV, text, and binary formats
- delimiters, headers, NULL representation, quoting, escaping, and encoding
- the difference between database-server paths and workstation paths

The course module extends those foundations with cloud-hosted PostgreSQL considerations, staging structures, views, and verification practices.


## Practice environments

This module uses two databases for different purposes.

### Pagila

Continue using Pagila for examples involving:

- exporting query results
- temporary tables
- views
- validation patterns

### ZoomZoom / `sqlda`

You will import the ZoomZoom dataset this week so it is available for later modules.

Treat the import as a data-engineering handoff:

> loading is not complete until validation is complete.


# Part 1: Why Data Movement Matters


## SQL analysis sits inside a larger pipeline

A database is rarely the first or last location of data.

Data may arrive as:

- CSV exports
- vendor extracts
- application logs
- public datasets
- spreadsheet exports
- machine-generated files
- data warehouse extracts

And SQL results may need to leave PostgreSQL for:

- reporting tools
- spreadsheets
- Python
- dashboards
- audits
- downstream systems


## Bulk movement versus row-by-row work

You already know `INSERT`.

```sql
INSERT INTO some_table (col1, col2)
VALUES ('A', 10);
```

That is appropriate for small, targeted changes.

For thousands or millions of rows, a bulk-loading mechanism is usually more efficient because it reduces repeated client/server communication and lets PostgreSQL use loading optimizations.


## The textbook's COPY perspective

The textbook presents `COPY` as a mechanism for moving data efficiently between PostgreSQL and external files.

Conceptually:

```text
table/query  ── COPY TO ──> external representation

external file ── COPY FROM ──> table
```

The direction is controlled by `TO` versus `FROM`.


## Quick check 1

**Think before opening the answer.**

Which direction moves data into PostgreSQL: `COPY ... TO` or `COPY ... FROM`?

<details>
<summary>Answer</summary>

`COPY ... FROM` moves data from an external source into a PostgreSQL table.

</details>


# Part 2: COPY TO — Exporting Data


## Export an entire table

A basic server-side export looks like:

```sql
COPY film
TO '/absolute/server/path/film.csv'
WITH (FORMAT CSV, HEADER TRUE);
```

Important:

- the path belongs to the **database server**
- PostgreSQL must have permission to write there
- cloud-hosted databases usually cannot write to arbitrary files on your laptop


## Export a query result

You can export the result of a `SELECT`, not only an entire table.

```sql
COPY (
    SELECT
        film_id,
        title,
        rating,
        rental_rate
    FROM film
    WHERE rating = 'G'
    ORDER BY title
)
TO '/absolute/server/path/g_films.csv'
WITH (FORMAT CSV, HEADER TRUE);
```

This is often more useful analytically because the exported file can contain only the rows and columns the recipient needs.


## Export to standard output

The textbook introduces:

```sql
COPY (
    SELECT *
    FROM products
    LIMIT 5
)
TO STDOUT
WITH CSV HEADER;
```

`STDOUT` sends the representation to standard output rather than directly to a named server-side file.


## Why headers matter

Without a header:

```text
1,ACADEMY DINOSAUR,PG,0.99
2,ACE GOLDFINGER,G,4.99
```

With a header:

```text
film_id,title,rating,rental_rate
1,ACADEMY DINOSAUR,PG,0.99
2,ACE GOLDFINGER,G,4.99
```

Column names make an exported file easier to interpret and reduce downstream ambiguity.


## Quick check 2

**Think before opening the answer.**

Why is exporting a query often safer than exporting `SELECT *`?

<details>
<summary>Answer</summary>

It makes the contract explicit: you control which rows, columns, filters, transformations, and ordering are delivered instead of exporting every current table column.

</details>


# Part 3: Server-Side COPY versus Client-Side \copy


## They look similar but run in different places

This distinction is one of the most important practical ideas in the module.

### `COPY`

```text
database server
     ↕
server filesystem
```

### `\copy`

```text
your workstation
     ↕
psql client
     ↕
database connection
     ↕
PostgreSQL server
```

The file path is interpreted by different machines.


## Server-side COPY

```sql
COPY film
TO '/server/path/film.csv'
WITH (FORMAT CSV, HEADER TRUE);
```

The PostgreSQL server process must be able to access that path.

If PostgreSQL is hosted in the cloud, your local `Downloads` folder is not part of the server filesystem.


## Client-side \copy

In `psql`:

```sql
\copy film TO '/local/path/film.csv' WITH (FORMAT CSV, HEADER TRUE)
```

`\copy` is a psql meta-command.

The client reads or writes the file and streams the data through the database connection.


## Why this matters for Neon

With a cloud database:

```text
your laptop ≠ Neon server filesystem
```

So a local path such as:

```text
C:\Users\YourName\Downloads\customers.csv
```

cannot normally be opened by server-side `COPY`.

Use:

- `\copy` through `psql`, or
- DBeaver's import/export tools

when the source or destination file is on your own machine.


## DBeaver as the graphical client

DBeaver can perform the same practical role as a client-side file-transfer tool.

For importing:

```text
table
  → right-click
  → Import Data
  → choose CSV
  → map columns
  → review settings
  → execute
```

For exporting:

```text
query result
  → Export Resultset
  → choose CSV
  → configure delimiter/header/encoding
```


## Quick check 3

**Think before opening the answer.**

Your PostgreSQL database is hosted on Neon and the CSV is in your laptop's Downloads folder. Which approach is appropriate: server-side COPY from that local path, or client-side `\copy`/DBeaver?

<details>
<summary>Answer</summary>

Client-side `\copy` or DBeaver. The Neon server cannot directly read a path on your laptop.

</details>


# Part 4: Configuring COPY and CSV


## FORMAT

Common formats include:

```sql
FORMAT CSV
FORMAT TEXT
FORMAT BINARY
```

For analytical interchange, CSV is the most common because it is broadly readable.

Binary format is useful for PostgreSQL-oriented workflows but is not a general-purpose interchange format for spreadsheets and typical analytics tools.


## DELIMITER

CSV normally uses a comma:

```sql
DELIMITER ','
```

But you may receive:

```text
A|B|C
```

Then specify:

```sql
DELIMITER '|'
```

The parser has to match the file's actual structure.


## HEADER

For export:

```sql
HEADER TRUE
```

writes column names.

For import:

```sql
HEADER TRUE
```

tells PostgreSQL or the client tool that the first file row is a header rather than data.


## NULL representation

Missing data is not always written the same way.

Files may contain:

```text
NULL
```

or:

```text
\N
```

or an empty field:

```text
value1,,value3
```

The import configuration must match the source file's convention.


## Empty string is not automatically the same as NULL

These two values have different meanings:

```text
''
```

and:

```text
NULL
```

An empty string is a known zero-length text value.

`NULL` means the value is missing or unknown.

If the file uses empty fields to mean missing data, configure the load deliberately rather than assuming.


## QUOTE

CSV fields may contain commas:

```text
"Smith, Jane",Toronto,CA
```

The quote character tells the parser that the comma inside the quoted text belongs to the value and is not a column separator.


## ESCAPE

Escape rules matter when text contains quote characters or other special characters.

A correctly configured CSV parser needs to distinguish:

```text
field delimiter
quote character
literal quote inside a value
row boundary
```

Incorrect quoting can shift values into the wrong columns.


## ENCODING

A file can be structurally valid but still fail or corrupt text if its character encoding does not match expectations.

For modern analytical workflows, UTF-8 is a common choice:

```sql
ENCODING 'UTF8'
```

Encoding deserves special attention when names, addresses, or text contain non-ASCII characters.


## Quick check 4

**Think before opening the answer.**

A person's surname contains a comma and the CSV parser treats it as two columns. Which COPY setting is most directly related to preserving the field as one value?

<details>
<summary>Answer</summary>

The CSV quoting rules, especially the `QUOTE` configuration and correct quoting in the file.

</details>


# Part 5: COPY FROM — Importing Data


## Target table first

Before importing a file, PostgreSQL needs a target structure.

Example staging table:

```sql
CREATE TABLE product_import (
    product_id       BIGINT,
    model            TEXT,
    model_year       INTEGER,
    product_type     TEXT,
    base_msrp        NUMERIC,
    production_start TIMESTAMP,
    production_end   TIMESTAMP
);
```

The table defines the types the incoming values must satisfy.


## Server-side COPY FROM

```sql
COPY product_import
FROM '/server/path/products.csv'
WITH (
    FORMAT CSV,
    HEADER TRUE
);
```

The server reads the file and converts each field into the target column type.


## Client-side \copy FROM

In `psql`:

```sql
\copy product_import FROM '/local/path/products.csv' WITH (FORMAT CSV, HEADER TRUE)
```

The local client reads the file and streams its contents to PostgreSQL.


## Match column order explicitly

A safer pattern is to name the columns:

```sql
\copy product_import (
    product_id,
    model,
    model_year,
    product_type,
    base_msrp,
    production_start,
    production_end
)
FROM '/local/path/products.csv'
WITH (FORMAT CSV, HEADER TRUE)
```

This documents the file-to-table mapping.


## Why type mismatches happen

Suppose the table expects:

```sql
model_year INTEGER
```

but the CSV contains:

```text
N/A
```

PostgreSQL cannot convert `"N/A"` into an integer.

Possible responses include:

- clean the source file
- load into a text-based staging table first
- map a sentinel to `NULL`
- transform values before inserting into the final typed table


## Staging table pattern

A robust workflow is:

```text
raw file
   ↓
staging table
   ↓
validation / cleaning
   ↓
typed permanent table
```

The staging table acts as a buffer between unpredictable external data and a trusted analytical schema.


## Example raw staging table

```sql
CREATE TEMP TABLE raw_product_import (
    product_id_raw TEXT,
    model_raw TEXT,
    year_raw TEXT,
    price_raw TEXT
);
```

Then validate before casting:

```sql
SELECT *
FROM raw_product_import
WHERE year_raw !~ '^[0-9]{4}$';
```

External data does not have to be trusted immediately.


# Part 6: Why COPY Is Efficient


## Fewer round trips

Imagine sending 100,000 individual commands:

```text
INSERT row 1
INSERT row 2
INSERT row 3
...
```

That creates repeated parsing, command handling, and client/server communication.

Bulk loading sends data in larger blocks and is optimized for the ingestion task.


## Performance is not the only benefit

Bulk loading also creates a clearer workflow:

```text
file
 → load
 → validate
 → commit/use
```

This makes the import auditable and repeatable.

In analytics, reproducibility is often more important than merely getting rows into a table once.


# Part 7: Common Import Failure Modes


## Failure 1: column count mismatch

Target:

```text
4 columns
```

File row:

```text
A,B,C,D,E
```

The parser cannot map five fields into four target columns.

Check:

- delimiter
- quotes
- target column list
- unexpected extra columns


## Failure 2: data type mismatch

Example:

```text
amount = "unknown"
```

Target:

```sql
amount NUMERIC
```

A strongly typed database rejects a value that cannot be converted.


## Failure 3: duplicate keys

If the target has:

```sql
PRIMARY KEY (customer_id)
```

and the incoming file repeats an existing key, the load may fail.

Investigate whether the file is:

- a full replacement
- an incremental append
- a duplicate extract
- intended for upsert logic


## Failure 4: encoding mismatch

Symptoms may include:

- load errors
- replacement characters
- mangled accented names
- corrupted symbols

Verify file encoding before treating the imported text as trustworthy.


## Failure 5: line-ending problems

Files created on different operating systems can use different row terminators.

Most modern tools handle this well, but hidden carriage-return characters can still appear in text values when parsing is misconfigured.

A field that visually looks like:

```text
Completed
```

may actually contain:

```text
Completed\r
```


## Failure 6: delimiter inside text

A malformed CSV row:

```text
Smith, Jane,Toronto,ON
```

may be interpreted as four fields instead of three.

Correct CSV quoting should preserve:

```text
"Smith, Jane",Toronto,ON
```


## Failure 7: wrong header setting

If the file has a header but the import does not skip it, PostgreSQL may try to load:

```text
customer_id
```

into an integer column.

If the file does not have a header but `HEADER TRUE` is enabled, the first real data row may be skipped.


## Failure 8: silent semantic errors

Some failures do not produce a database error.

Examples:

- dollars imported as cents
- day/month dates interpreted incorrectly
- category labels containing trailing spaces
- duplicated business records with different IDs
- row counts matching while values are wrong

This is why successful execution is only the beginning of validation.


## Quick check 5

**Think before opening the answer.**

Why can an import that reports no SQL error still be analytically wrong?

<details>
<summary>Answer</summary>

Because many semantic problems satisfy the table's technical constraints: wrong units, misinterpreted dates, duplicated business records, malformed labels, or incorrect but type-compatible values can all load successfully.

</details>


# Part 8: Temporary Tables


## Temporary table

A temporary table is a table scoped to your current database session.

```sql
CREATE TEMP TABLE recent_payments AS
SELECT *
FROM payment
WHERE payment_date >= DATE '2005-08-01';
```

Within that session, you can query it like another table.


## Temporary tables store rows

```sql
SELECT COUNT(*)
FROM recent_payments;
```

The result of the defining query was materialized into temporary table storage.

Unlike a standard view, PostgreSQL does not rerun the original `SELECT` each time you query the temp table.


## Typical uses

Temporary tables are useful when you need:

- import staging
- intermediate calculations reused repeatedly
- a smaller subset for testing
- a scratch area for multi-step SQL
- an indexed intermediate structure


## Session lifetime

A temporary table usually disappears when the session ends.

That means:

- it is useful for scratch work
- other sessions normally cannot see your temp table
- reconnecting creates a new session, so the old temp table is gone


## Explicit cleanup

Even though the table will eventually disappear:

```sql
DROP TABLE IF EXISTS recent_payments;
```

can make a long script easier to rerun predictably.


# Part 9: Views


## A view stores query logic

```sql
CREATE VIEW open_rentals AS
SELECT
    rental_id,
    customer_id,
    inventory_id,
    rental_date
FROM rental
WHERE return_date IS NULL;
```

A standard view behaves like a named query.


## Query the view

```sql
SELECT
    customer_id,
    COUNT(*) AS items_out
FROM open_rentals
GROUP BY customer_id
ORDER BY items_out DESC;
```

The view gives a reusable business concept to later queries.


## A standard view does not normally store the result rows

When you query:

```sql
SELECT *
FROM open_rentals;
```

PostgreSQL uses the view definition against the current underlying tables.

If the base data changes, the next view query reflects those changes.


## Why use a view?

A view can:

- hide repetitive joins
- standardize a business definition
- simplify BI queries
- expose selected columns
- support permissions and abstraction
- make analytical SQL easier to read


## Drop a view

```sql
DROP VIEW IF EXISTS open_rentals;
```

Views persist until removed, unlike ordinary temporary tables.


# Part 10: Temporary Table vs View vs Materialized View


## Comparison

| Structure | Stores rows? | Lifetime | Typical use |
|---|---:|---|---|
| Temporary table | Yes | session | staging, scratch, reusable intermediate data |
| Standard view | No result storage | persistent | reusable query logic |
| Materialized view | Yes | persistent until refreshed | cached expensive result |
| Permanent table | Yes | persistent | authoritative stored data |


## Materialized view

A materialized view stores the query result:

```sql
CREATE MATERIALIZED VIEW monthly_payment_summary AS
SELECT
    DATE_TRUNC('month', payment_date) AS month,
    SUM(amount) AS revenue
FROM payment
GROUP BY DATE_TRUNC('month', payment_date);
```

The stored result becomes stale when source data changes.


## Refresh a materialized view

```sql
REFRESH MATERIALIZED VIEW monthly_payment_summary;
```

Materialized views exchange freshness for faster repeated access.

They are not the main focus of this module, but understanding the distinction prevents confusion with ordinary views.


## Quick check 6

**Think before opening the answer.**

Which structure is best when you need a session-only staging area that physically stores imported rows?

<details>
<summary>Answer</summary>

A temporary table.

</details>


## Quick check 7

**Think before opening the answer.**

Which structure is best when several sessions should reuse the same SELECT logic and see current underlying data?

<details>
<summary>Answer</summary>

A standard view.

</details>


# Part 11: A Professional Import Workflow


## Step 1: inspect the source before loading

Before import, answer:

- What is the delimiter?
- Is there a header?
- What represents missing values?
- What is the encoding?
- How are dates formatted?
- Are IDs unique?
- What units are numeric fields using?
- How many rows are expected?


## Step 2: inspect the target DDL

The DDL tells you:

- column names
- column order
- data types
- primary keys
- foreign keys
- nullable columns
- default values
- constraints

Import success depends on matching the file to that contract.


## Step 3: load

Choose the correct mechanism:

```text
local file + cloud database
        ↓
DBeaver or \copy
```

For a local PostgreSQL server with file access, server-side `COPY` may also be appropriate.


## Step 4: validate row counts

```sql
SELECT COUNT(*)
FROM imported_table;
```

A row count can detect:

- empty loads
- obvious truncation
- unexpected duplicate appends

But row count alone cannot prove correctness.


## Step 5: validate structure

```sql
SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'customers'
ORDER BY ordinal_position;
```

Compare the actual database structure with the supplied DDL.


## Step 6: validate keys

```sql
SELECT COUNT(*) AS null_primary_keys
FROM customers
WHERE customer_id IS NULL;
```

Then check duplicates:

```sql
SELECT
    customer_id,
    COUNT(*)
FROM customers
GROUP BY customer_id
HAVING COUNT(*) > 1;
```


## Step 7: validate ranges

```sql
SELECT
    MIN(date_added) AS min_date,
    MAX(date_added) AS max_date
FROM customers;
```

For numeric measures:

```sql
SELECT
    MIN(base_msrp),
    MAX(base_msrp),
    AVG(base_msrp)
FROM products;
```

Extreme values can reveal unit, parsing, or source problems.


## Step 8: validate categorical distributions

```sql
SELECT
    product_type,
    COUNT(*)
FROM products
GROUP BY product_type
ORDER BY COUNT(*) DESC;
```

Unexpected categories can reveal:

- whitespace
- mixed casing
- malformed values
- coding changes


## Step 9: sample raw rows

```sql
SELECT *
FROM customers
ORDER BY customer_id
LIMIT 20;
```

Human inspection is not sufficient for full validation, but it is useful for catching obvious mapping errors quickly.


## Step 10: document the validation

A professional import record should capture:

- source filename/version
- load date
- target table
- row count
- validation queries
- exceptions
- transformations applied
- who approved the data for use


# Part 12: ZoomZoom Import Readiness


## Why ZoomZoom matters

The `sqlda` dataset becomes the working environment for later analytical modules.

Typical ZoomZoom domains include:

- customers
- products
- sales
- salespeople/dealerships
- marketing/email activity
- location fields

A bad import now can create misleading results later.


## Do not assume table names

Run:

```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
```

Compare the result with the `create_tables.sql` script supplied with the dataset.


## Row-count validation pattern

Once you know the actual table names:

```sql
SELECT 'customers' AS table_name,
       COUNT(*) AS row_count
FROM customers

UNION ALL

SELECT 'products',
       COUNT(*)
FROM products

UNION ALL

SELECT 'sales',
       COUNT(*)
FROM sales

ORDER BY table_name;
```

Adapt the list to the actual imported schema.


## Column validation pattern

```sql
SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'sales'
ORDER BY ordinal_position;
```

This confirms that the imported table is structurally usable for later analysis.


## Referential integrity spot checks

If a sales row contains a `customer_id`, ask whether that customer exists.

Example:

```sql
SELECT COUNT(*) AS unmatched_customer_rows
FROM sales s
LEFT JOIN customers c
  ON s.customer_id = c.customer_id
WHERE s.customer_id IS NOT NULL
  AND c.customer_id IS NULL;
```

A nonzero result deserves investigation.


## Validation is cumulative

A trustworthy load passes several different tests:

```text
tables exist
   +
row counts plausible
   +
types correct
   +
keys valid
   +
ranges sensible
   +
relationships plausible
   +
sample rows readable
```

No single query proves the dataset is correct.


# Part 13: Real-World Import Scenarios


## Scenario 1: Payroll extract

A payroll CSV arrives weekly.

You should validate:

- employee IDs
- pay period dates
- duplicate rows
- hours range
- negative or impossible values
- total payroll amount
- missing departments

A correct row count alone is not enough.


## Scenario 2: CRM export

A CRM file may contain:

- repeated customers
- inconsistent phone formats
- blank emails
- unexpected country codes
- new category labels

Load into staging, profile the values, then standardize before analytics.


## Scenario 3: Healthcare encounters

Key concerns might include:

- protected identifiers
- encounter uniqueness
- timestamp order
- invalid provider IDs
- impossible age values
- missing required fields

The more consequential the analysis, the stronger the import validation should be.


## Scenario 4: Financial transactions

Validation should include:

- duplicate transaction IDs
- currency consistency
- amount sign rules
- timezone handling
- settlement versus transaction date
- account referential integrity

A syntactically successful import can still create materially wrong totals.


# Part 14: Export Design


## Export only what is needed

Instead of:

```sql
SELECT *
FROM customer;
```

prefer an explicit result:

```sql
SELECT
    customer_id,
    first_name,
    last_name,
    email
FROM customer
WHERE active = 1;
```

Explicit exports are easier to govern and less likely to leak irrelevant fields.


## Make the export reproducible

A reproducible export has:

- saved SQL
- explicit columns
- explicit filters
- defined sort order when needed
- documented date range
- known data source/version


## Example analytical export

In `psql`:

```sql
\copy (
    SELECT
        customer_id,
        SUM(amount) AS total_paid
    FROM payment
    GROUP BY customer_id
    ORDER BY total_paid DESC
)
TO '/local/path/customer_totals.csv'
WITH (FORMAT CSV, HEADER TRUE)
```

This produces an analysis-ready output rather than a raw table dump.


# Part 15: Views as Export Helpers


## Simplify repeated export logic

A long export query can be encoded as a view:

```sql
CREATE VIEW customer_payment_summary AS
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    COUNT(p.payment_id) AS payment_count,
    SUM(p.amount) AS total_paid
FROM customer c
LEFT JOIN payment p
  ON c.customer_id = p.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name;
```

Then export the view.


## Why this pattern helps

It separates two concerns:

```text
business logic
    ↓
view

file-transfer logic
    ↓
\copy / export tool
```

That improves readability and reuse.


# Part 16: pg_dump Awareness


## COPY is not a full database backup

`COPY` is primarily about data movement in table-oriented or query-oriented formats.

Database backup and restoration are different problems.

PostgreSQL provides tools such as:

- `pg_dump`
- `pg_dumpall`
- `pg_restore`


## Analytical export versus backup

### CSV export

Good for:

- Excel
- Python
- BI tools
- data sharing

### Database dump

Good for:

- recreating PostgreSQL objects
- database migration
- backup/restore workflows

A dump archive is not designed to be a general spreadsheet-style interchange file.


# Part 17: Guided Practice


## Practice 1: export G-rated films

Write a `\copy` command that exports:

- film ID
- title
- rating
- rental rate

for G-rated films to a local CSV with a header.


<details>
<summary>One solution</summary>

```sql
\copy (
    SELECT
        film_id,
        title,
        rating,
        rental_rate
    FROM film
    WHERE rating = 'G'
    ORDER BY title
)
TO '/local/path/g_films.csv'
WITH (FORMAT CSV, HEADER TRUE)
```

Use a path appropriate for your operating system and local workstation.

</details>


## Practice 2: import into a staging table

Suppose `incoming.csv` contains:

```text
customer_id,email
1,a@example.com
2,b@example.com
```

Write the table DDL and client-side import.


<details>
<summary>One solution</summary>

```sql
CREATE TEMP TABLE customer_email_staging (
    customer_id INTEGER,
    email TEXT
);
```

Then in psql:

```sql
\copy customer_email_staging (customer_id, email)
FROM '/local/path/incoming.csv'
WITH (FORMAT CSV, HEADER TRUE)
```

</details>


## Practice 3: validate duplicates

Write a query that reports any duplicate customer IDs in a staging table called `customer_staging`.


<details>
<summary>Solution</summary>

```sql
SELECT
    customer_id,
    COUNT(*) AS occurrences
FROM customer_staging
GROUP BY customer_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC, customer_id;
```

</details>


## Practice 4: validate categories

Write a query that shows every distinct `product_type` and its frequency.


<details>
<summary>Solution</summary>

```sql
SELECT
    product_type,
    COUNT(*) AS row_count
FROM products
GROUP BY product_type
ORDER BY row_count DESC, product_type;
```

</details>


## Practice 5: temporary summary

Create a temporary table containing total payments by customer and return the ten highest totals.


<details>
<summary>Solution</summary>

```sql
CREATE TEMP TABLE customer_totals AS
SELECT
    customer_id,
    SUM(amount) AS total_paid
FROM payment
GROUP BY customer_id;
```

Then:

```sql
SELECT *
FROM customer_totals
ORDER BY total_paid DESC
LIMIT 10;
```

</details>


## Practice 6: create a view

Create a view named `active_customers` that returns customer ID, first name, last name, and email for active customers.


<details>
<summary>Solution</summary>

```sql
CREATE VIEW active_customers AS
SELECT
    customer_id,
    first_name,
    last_name,
    email
FROM customer
WHERE active = 1;
```

</details>


# Part 18: AI Critique and Import Review


## AI can generate syntactically plausible but unsafe import advice

Review generated data-loading instructions for:

- server versus client path confusion
- incorrect delimiter assumptions
- missing `HEADER`
- wrong target-column order
- data-type mismatch
- failure to validate after load
- destructive `TRUNCATE` suggestions
- unsupported assumptions about permissions


## AI critique exercise 1

A model says:

> "Because your Neon database is connected to DBeaver on your laptop, run `COPY customers FROM 'C:\Users\me\Downloads\customers.csv'`."

What is wrong?


<details>
<summary>Critique</summary>

The PostgreSQL server runs in the cloud and cannot directly see the laptop's filesystem. Use DBeaver's import functionality or client-side `\copy` from psql instead.

</details>


## AI critique exercise 2

A model recommends:

```sql
SELECT COUNT(*)
FROM customers;
```

and says:

> "If the count is correct, the import is fully verified."

What is missing?


<details>
<summary>Critique</summary>

A correct row count does not verify column mapping, data types, key uniqueness, NULL behavior, value ranges, category consistency, referential integrity, units, or semantic correctness. Row count is one validation step, not proof of trustworthiness.

</details>


## AI critique exercise 3

A model suggests adding `DISTINCT` to every validation query to "remove duplicate problems."

Why is this a bad validation strategy?


<details>
<summary>Critique</summary>

`DISTINCT` can hide duplicates rather than diagnose them. During validation, duplicates are often evidence you need to inspect. Measure and investigate them explicitly instead of suppressing them.

</details>


# Part 19: Mid-Course Project Connection


## Module 9 is also a checkpoint

The mid-course project uses the analytical skills developed before this import module:

- filters
- joins
- subqueries
- CTEs
- set operations
- transformation functions
- aggregation
- window functions


## Data movement and analytical reasoning are related

The same habits apply to both:

### Query design
State the intended grain.

### Import design
State the expected row and column contract.

### Query validation
Check counts, joins, and ranges.

### Import validation
Check counts, keys, types, and ranges.

In both cases, "the command ran" is not enough.


## EXPLAIN ANALYZE awareness

The mid-course project asks you to inspect query execution.

Remember:

```sql
EXPLAIN ANALYZE
SELECT ...;
```

actually executes the query and reports the plan with observed timing/row information.

Use it on safe `SELECT` queries for the assignment.


# Part 20: Discussion Preparation


## Reflect on actual verification

A strong discussion response should explain what you really checked.

For example:

> "I compared imported row counts to the expected source counts, inspected the `information_schema.columns` metadata for the sales table, and checked primary-key nulls. The row counts matched, but one text category contained trailing whitespace, which I detected through a grouped frequency query."

Specific evidence is stronger than saying:

> "I verified the data."


## Connect validation to earlier SQL concepts

Possible connections include:

- NULL checks ↔ Module 2
- data types ↔ Module 3
- joins and foreign keys ↔ Module 4
- CTE-based staging ↔ Module 5
- cleaning functions ↔ Module 6
- grouped distributions ↔ Module 7
- ranking or sequence checks ↔ Module 8


## A professional trust question

Before using newly imported data in a decision, ask:

> What failure could produce a plausible result without producing an SQL error?

That question leads to stronger validation than checking only for failed commands.


# Part 21: Knowledge Checks


## Knowledge check 1

**Think before opening the answer.**

Where does server-side `COPY` read or write files?

<details>
<summary>Answer</summary>

On the PostgreSQL server's filesystem.

</details>


## Knowledge check 2

**Think before opening the answer.**

Where does `\copy` read or write files?

<details>
<summary>Answer</summary>

On the client workstation where psql is running.

</details>


## Knowledge check 3

**Think before opening the answer.**

Which keyword indicates data is being loaded into a table?

<details>
<summary>Answer</summary>

`FROM`.

</details>


## Knowledge check 4

**Think before opening the answer.**

Which keyword indicates data is being exported?

<details>
<summary>Answer</summary>

`TO`.

</details>


## Knowledge check 5

**Think before opening the answer.**

What does `HEADER TRUE` do on CSV import?

<details>
<summary>Answer</summary>

It tells the loader that the first file row is a header rather than a data row.

</details>


## Knowledge check 6

**Think before opening the answer.**

What is a primary reason to use a staging table?

<details>
<summary>Answer</summary>

To isolate raw external data so it can be inspected, cleaned, validated, and converted before entering trusted permanent tables.

</details>


## Knowledge check 7

**Think before opening the answer.**

Does a standard view store a separate copy of its result rows?

<details>
<summary>Answer</summary>

No. A standard view stores query logic and evaluates it against the underlying tables when queried.

</details>


## Knowledge check 8

**Think before opening the answer.**

Does a temporary table normally survive reconnecting to a new database session?

<details>
<summary>Answer</summary>

No. It is normally session-scoped and disappears when the session ends.

</details>


## Knowledge check 9

**Think before opening the answer.**

Why is a correct row count insufficient to validate an import?

<details>
<summary>Answer</summary>

Because values can still be mapped incorrectly, duplicated semantically, assigned wrong units, parsed into wrong dates, corrupted textually, or violate business expectations while preserving the expected row count.

</details>


## Knowledge check 10

**Think before opening the answer.**

When loading a local CSV into a cloud-hosted Neon database, what should you normally use?

<details>
<summary>Answer</summary>

A client-side mechanism such as psql `\copy` or DBeaver's import tool.

</details>


# Part 22: Concept Maps


## Data movement map

```text
LOCAL / EXTERNAL FILE
        |
        |  \copy / DBeaver
        v
POSTGRESQL TABLE
        |
        |  validate
        v
TRUSTED ANALYTICAL DATA
        |
        |  query / transform
        v
RESULT SET
        |
        |  \copy / DBeaver export
        v
EXTERNAL CSV / TOOL
```


## Structure selection map

```text
Need permanent authoritative rows?
        → permanent table

Need session-only scratch/staging rows?
        → TEMP TABLE

Need reusable live query logic?
        → VIEW

Need persistent cached query output?
        → MATERIALIZED VIEW
```


## Validation map

```text
IMPORT COMPLETES
      |
      v
tables exist?
      |
      v
row counts plausible?
      |
      v
types correct?
      |
      v
keys valid?
      |
      v
NULLs expected?
      |
      v
ranges sensible?
      |
      v
relationships valid?
      |
      v
categories clean?
      |
      v
sample rows sensible?
      |
      v
DATA READY FOR ANALYSIS
```


# Part 23: ZoomZoom Import Checklist


## Before the import

Confirm you have:

- all CSV files
- `create_tables.sql`
- an active Neon connection
- a local folder with known file paths
- enough time to validate after loading


## During the import

For each table:

1. create the schema/table
2. choose the correct source CSV
3. verify column mapping
4. verify header setting
5. verify delimiter
6. verify NULL settings
7. execute the import
8. record errors rather than bypassing them


## After the import

Run:

```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
```

Then:

- row counts
- column metadata
- primary-key checks
- duplicate checks
- range checks
- category checks
- referential-integrity checks


## Do not continue with bad data

If validation fails:

```text
stop
  ↓
identify the failure
  ↓
correct load/mapping/source
  ↓
reload if necessary
  ↓
rerun validation
```

Do not compensate for a broken import later with increasingly complicated analytical SQL.


# Part 24: Module Summary


## Key takeaways

- `COPY` is PostgreSQL's bulk data-movement mechanism.
- `COPY ... TO` exports; `COPY ... FROM` imports.
- Server-side `COPY` uses paths visible to the database server.
- psql `\copy` uses paths visible to the client workstation.
- In cloud-hosted Neon workflows, local files should normally be moved with `\copy` or DBeaver.
- CSV configuration must match the real file: format, delimiter, header, NULL representation, quote rules, and encoding.
- Bulk loading is efficient, but speed does not replace validation.
- Temporary tables are useful session-scoped staging and scratch structures.
- Standard views persist query logic without normally storing result rows.
- Materialized views persist calculated rows and require refresh.
- Newly imported data should be treated as untrusted until structure, counts, keys, NULLs, ranges, categories, and relationships are checked.
- The ZoomZoom import is foundational for the next phase of the course.


## The habit to carry forward

Before using any newly loaded dataset, be able to answer:

> **What evidence do I have that the data was loaded completely, mapped correctly, and still means what the source intended?**

That is the difference between loading data and establishing analytical trust.


## Up next: Module 10

Module 10 moves into more complex PostgreSQL data types.

You will begin working with data that does not always fit neatly into ordinary scalar columns, including:

- JSON
- JSONB
- arrays
- nested values
- extraction operators and functions

The ZoomZoom database imported in this module becomes the foundation for that work.


## References

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for data analytics: Analyze data effectively, uncover insights and master advanced SQL for real-world applications* (4th ed.). Packt Publishing.

PostgreSQL Global Development Group. (n.d.). *COPY*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *psql*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *CREATE TABLE*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *CREATE VIEW*. PostgreSQL 16 Documentation.
